# Research Library Repository Chroma Query

In [1]:
%reload_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import os
from collections import Counter
from datetime import datetime, timezone


sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../..')))

from apps.agentic.core.document_loaders.research_library_document_loader import ResearchLibraryChromaDocumentLoader
from apps.agentic.core.constants import RESEARCH_LIBRARY_DB_NAME, RESEARCH_LIBRARY_COLLECTION_NAME
from apps.agentic.core.utils import (set_chatgpt_env, set_langsmith_env, set_tavily_env, 
                                     set_github_env)

DB_PATH = Path("../../.db").resolve()

set_chatgpt_env(filedir="../../.keys")
set_langsmith_env(filedir="../../.keys")

## Verify Document Counts

In [22]:
RESEARCH_LIBRARY_DB_NAME, RESEARCH_LIBRARY_COLLECTION_NAME, DB_PATH, os.listdir(DB_PATH)

('research_library',
 'research-library',
 '../../.db',
 ['.DS_Store', 'research_library', 'pdf_document_library', 'github'])

In [34]:
vs = ResearchLibraryChromaDocumentLoader(DB_PATH).vectorstore
coll = vs._collection

In [35]:
client = vs._client
print([c.name for c in client.list_collections()])
print("Opened:", coll.name)
print("total docs:", coll.count())

['research-library']
Opened: research-library
total docs: 1258


## Verify Document Metadata

In [36]:
probe = coll.get(limit=5000)
metas  = [m for m in probe.get("metadatas", []) if m]
len(metas)

1258

In [37]:
keys = set().union(*(m.keys() for m in metas)) if metas else set()
for key in sorted(keys):
    print(key)

authors
date
ext
filename
h1
h2
images
path
section
section_char_offset
start_index
tags
title
topic


In [38]:
print(Counter(m.get("title") for m in metas if "title" in m and m.get("title")))
print("authors counts:", Counter(m.get("authors") for m in metas if "authors" in m and m.get("authors")))

Counter({'Autoregressive Processes': 278, 'Fractional Brownian Motion': 256, 'Discrete Models of Financial Markets': 250, 'Analytic Mechanics': 174, 'The Evolution of Carnot’s Principle': 104, 'Bayesian Data Analysis': 102, 'Functions': 66, 'Brownian Motion': 16, 'Geometry': 12})
authors counts: Counter({'Troy Stribling': 1154, 'E. T. Jaynes': 104})


## Search Examples 

In [18]:
vs = ResearchLibraryChromaDocumentLoader(DB_PATH).vectorstore
retriever = vs.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 8,
        "fetch_k": 60,
        "filter": {"$and": [{"authors": "Troy Stribling"}, {"title": "Analytic Mechanics"}]},
    },
)

hits = retriever.invoke("How are Lagrange multipliers defined?")
[print(h) for h in hits]

page_content='The method of Lagrange multipliers used to determithe the constraing forces acting on a system.  
Consider a system described by $n$ generalized coordinates $q_{1}, q_{2}, \ldots, q_{n}$. Furthure assume that the system has $k$ constraints acting on it. The constraints eleminate $k$ of the generalized coordinates leaving a total $n-k$ linearly independent coordinates.
The time behavior of the system is described by Hamilton's Principal  
$$
\delta I=\delta \int_{t_{s}}^{t_{f}} L d t=\int_{t_{s}}^{t_{f}} \delta L d t=0
$$  
with $k$ equations defining holonomic constraints,  
$$
f_{j}\left(q_{1}, q_{2}, \ldots, q_{n}\right)=0 \quad \forall \quad j=1,2, \ldots, k
$$  
consider the variation of the constraints  
$$
\delta f_{j}=\sum_{l=1}^{n} \frac{\partial f_{j}}{\partial q_{l}} \delta q_{l}=0
$$  
For each $f_{j}$ define a scalar parameter $\lambda_{j}$ such that  
$$
\lambda_{j} \delta f_{j}=\sum_{l=1}^{n} \lambda_{j} \frac{\partial f_{j}}{\partial q_{l}} \delta q_{l}=0
$

[None, None, None, None, None, None, None, None]

In [19]:
vs = ResearchLibraryChromaDocumentLoader(DB_PATH).vectorstore
retriever = vs.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 8,
        "fetch_k": 60,
        "filter": {"$and": [{"authors": "E. T. Jaynes"}, {"title": "The Evolution of Carnot’s Principle"}]},
    },
)
hits = retriever.invoke("What did Willard Gibbs contribute to the concept of entropy?")
[print(h) for h in hits]

page_content='Another of the curiosities of this field is that, having done so much with entropy and demonstrated such a deep understanding of the logic underlying the second law, giving thermodynamics an entirely different character, Gibbs said almost nothing about what entropy really means. He showed, far more than anyone else, how much we can accomplish by maximizing entropy. Yet we cannot learn from Gibbs: "What are we actually doing when we maximize entropy?" For this we must turn to Boltzmann.' metadata={'topic': 'Review of historical development of the Second Law of Thermodynamics.', 'authors': 'E. T. Jaynes', 'section_char_offset': 7384, 'tags': 'jaynes,thermodynamics', 'ext': '.md', 'title': 'The Evolution of Carnot’s Principle', 'path': '/Users/troy/Develop/gly.fish/yada/research_library/Jaynes/carnot-8d44967b-c078-4902-980c-f834d26088a5.md', 'images': '', 'start_index': 7350, 'h1': "THE EVOLUTION OF CARNOT'S PRINCIPLE*", 'h2': '5. THIRD METAMORPHOSIS: GIBBS', 'filename': 'ca

[None, None, None, None, None, None, None, None]

In [20]:
vs = ResearchLibraryChromaDocumentLoader(DB_PATH).vectorstore
retriever = vs.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 8,
        "fetch_k": 60,
    },
)
hits = retriever.invoke("What did Willard Gibbs contribute to the concept of entropy?")
[print(h) for h in hits]

page_content='Another of the curiosities of this field is that, having done so much with entropy and demonstrated such a deep understanding of the logic underlying the second law, giving thermodynamics an entirely different character, Gibbs said almost nothing about what entropy really means. He showed, far more than anyone else, how much we can accomplish by maximizing entropy. Yet we cannot learn from Gibbs: "What are we actually doing when we maximize entropy?" For this we must turn to Boltzmann.' metadata={'topic': 'Review of historical development of the Second Law of Thermodynamics.', 'section_char_offset': 7384, 'h2': '5. THIRD METAMORPHOSIS: GIBBS', 'tags': 'jaynes,thermodynamics', 'section': 6, 'ext': '.md', 'path': '/Users/troy/Develop/gly.fish/yada/research_library/Jaynes/carnot-8d44967b-c078-4902-980c-f834d26088a5.md', 'start_index': 7350, 'h1': "THE EVOLUTION OF CARNOT'S PRINCIPLE*", 'images': '', 'filename': 'carnot-8d44967b-c078-4902-980c-f834d26088a5.md', 'title': 'The 

[None, None, None, None, None, None, None, None]